In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import DateType
from delta.tables import DeltaTable

# Get the insert_timestamp parameter
insert_ts = dbutils.widgets.get("insert_timestamp")

# Determine load type
load_type = "FULL" if not insert_ts or insert_ts.strip() == "" else "INCREMENTAL"
print(f"Load Type: {load_type}")
if load_type == "INCREMENTAL":
    print(f"Loading data where insert_timestamp > {insert_ts}")

Load Type: INCREMENTAL
Loading data where insert_timestamp > 2026-06-24T12:09:30.082+00:00


In [0]:
# Read from bronze table with conditional logic
if load_type == "FULL":
    # Full load - read all data
    bronze_df = spark.table("data_modelling_demo.bronze.scd2")
    print(f"Full load: Read {bronze_df.count()} records from bronze table")
else:
    # Incremental load - read only new data
    bronze_df = spark.table("data_modelling_demo.bronze.scd2") \
        .filter(F.col("insert_timestamp") > insert_ts)
    print(f"Incremental load: Read {bronze_df.count()} records from bronze table")

# Display sample data
display(bronze_df.limit(5))

Incremental load: Read 3 records from bronze table


customer_id,first_name,email,city,registration_date,last_updated_ts,_rescued_data,insert_timestamp
C001,john,john.smith@gmail.com,chicago,10-Jan-2024,2024-02-01 09:30:00,null,2026-06-25T12:22:41.805Z
C002,alice,alice123@gmail.com,boston,11-Jan-2024,2024-02-01 10:45:00,null,2026-06-25T12:22:41.805Z
C006,emma,emma.thomas@gmail.com,san francisco,01-Feb-2024,2024-02-01 11:15:00,null,2026-06-25T12:22:41.805Z


In [0]:
# Apply transformations
transformed_df = bronze_df \
    .withColumn("first_name", F.initcap(F.col("first_name"))) \
    .withColumn("email", F.lower(F.col("email"))) \
    .withColumn("registration_date", F.to_date(F.col("registration_date"), "dd-MMM-yyyy"))

print(f"Transformations applied successfully")
display(transformed_df.limit(5))

Transformations applied successfully


customer_id,first_name,email,city,registration_date,last_updated_ts,_rescued_data,insert_timestamp
C001,John,john.smith@gmail.com,chicago,2024-01-10,2024-02-01 09:30:00,null,2026-06-25T12:22:41.805Z
C002,Alice,alice123@gmail.com,boston,2024-01-11,2024-02-01 10:45:00,null,2026-06-25T12:22:41.805Z
C006,Emma,emma.thomas@gmail.com,san francisco,2024-02-01,2024-02-01 11:15:00,null,2026-06-25T12:22:41.805Z


In [0]:
# Apply data quality checks
from pyspark.sql import DataFrame

def apply_data_quality_checks(df: DataFrame) -> DataFrame:
    """
    Apply data quality checks and filter out invalid records
    """
    # Count before quality checks
    count_before = df.count()
    
    # Data quality rules:
    # 1. Remove records with null customer_id (primary key)
    # 2. Remove records with null or empty first_name
    # 3. Remove records with invalid email (must contain @)
    # 4. Remove records with null registration_date
    
    quality_df = df \
        .filter(F.col("customer_id").isNotNull()) \
        .filter((F.col("first_name").isNotNull()) & (F.trim(F.col("first_name")) != "")) \
        .filter((F.col("email").isNotNull()) & (F.col("email").contains("@"))) \
        .filter(F.col("registration_date").isNotNull())
    
    # Count after quality checks
    count_after = quality_df.count()
    records_rejected = count_before - count_after
    
    print(f"Data Quality Check Results:")
    print(f"  Records before: {count_before}")
    print(f"  Records after: {count_after}")
    print(f"  Records rejected: {records_rejected}")
    
    return quality_df

# Apply quality checks
quality_df = apply_data_quality_checks(transformed_df)
display(quality_df.limit(5))

Data Quality Check Results:
  Records before: 3
  Records after: 3
  Records rejected: 0


customer_id,first_name,email,city,registration_date,last_updated_ts,_rescued_data,insert_timestamp
C001,John,john.smith@gmail.com,chicago,2024-01-10,2024-02-01 09:30:00,null,2026-06-25T12:22:41.805Z
C002,Alice,alice123@gmail.com,boston,2024-01-11,2024-02-01 10:45:00,null,2026-06-25T12:22:41.805Z
C006,Emma,emma.thomas@gmail.com,san francisco,2024-02-01,2024-02-01 11:15:00,null,2026-06-25T12:22:41.805Z


In [0]:
# Define silver table name
silver_table = "data_modelling_demo.silver.scd2"

# Add SCD Type 2 columns
current_timestamp = F.current_timestamp()
final_df = quality_df \
    .withColumn("processed_timestamp", current_timestamp) \
    .withColumn("effective_start_date", current_timestamp) \
    .withColumn("effective_end_date", F.lit(None).cast("timestamp")) \
    .withColumn("is_current", F.lit(True))

# Check if silver table exists
if spark.catalog.tableExists(silver_table):
    print(f"Silver table '{silver_table}' exists. Performing SCD Type 2 MERGE...")
    
    # Load existing silver table as Delta table
    silver_delta = DeltaTable.forName(spark, silver_table)
    
    # SCD Type 2: Track historical changes
    # Step 1: Close out existing current records that have changed
    silver_delta.alias("target").merge(
        final_df.alias("source"),
        "target.customer_id = source.customer_id AND target.is_current = true"
    ).whenMatchedUpdate(
        condition = """
            target.first_name <> source.first_name OR
            target.email <> source.email OR
            target.city <> source.city OR
            target.registration_date <> source.registration_date OR
            target.last_updated_ts <> source.last_updated_ts
        """,
        set = {
            "effective_end_date": "source.effective_start_date",
            "is_current": "false"
        }
    ).execute()
    
    # Step 2: Insert new versions for changed records and brand new records
    # Get records that either don't exist or have changed
    existing_current_df = spark.table(silver_table).filter(F.col("is_current") == True)
    
    # Left anti join to find completely new customer_ids
    new_customers_df = final_df.join(
        existing_current_df.select("customer_id"),
        "customer_id",
        "left_anti"
    )
    
    # Inner join to find changed records (where effective_end_date was just updated)
    changed_customers_df = final_df.alias("source").join(
        existing_current_df.alias("target"),
        (F.col("source.customer_id") == F.col("target.customer_id")) &
        (
            (F.col("target.first_name") != F.col("source.first_name")) |
            (F.col("target.email") != F.col("source.email")) |
            (F.col("target.city") != F.col("source.city")) |
            (F.col("target.registration_date") != F.col("source.registration_date")) |
            (F.col("target.last_updated_ts") != F.col("source.last_updated_ts"))
        ),
        "inner"
    ).select("source.*")
    
    # Union new and changed records
    records_to_insert = new_customers_df.union(changed_customers_df)
    
    # Insert new versions
    if records_to_insert.count() > 0:
        records_to_insert.write \
            .format("delta") \
            .mode("append") \
            .saveAsTable(silver_table)
        
        print("✓ MERGE completed successfully (SCD Type 2)")
        print("  - Changed records: Previous version closed, new version inserted")
        print("  - New records: INSERTED with is_current=True")
        print("  - Unchanged records: Left as-is")
    else:
        print("✓ No new or changed records to insert")
else:
    print(f"Silver table '{silver_table}' does not exist. Creating new table...")
    
    # Create new silver table
    final_df.write \
        .format("delta") \
        .mode("overwrite") \
        .saveAsTable(silver_table)
    
    print(f"✓ Silver table '{silver_table}' created successfully with SCD Type 2 structure")

# Show table statistics
print(f"\nSilver table record count:")
silver_count = spark.table(silver_table).count()
current_count = spark.table(silver_table).filter(F.col("is_current") == True).count()
print(f"Total records in silver: {silver_count}")
print(f"Current records (is_current=True): {current_count}")
print(f"Historical records: {silver_count - current_count}")

print("\nSample from silver table (showing current and historical):")
display(spark.table(silver_table).orderBy(F.col("customer_id"), F.col("effective_start_date").desc()).limit(20))

Silver table 'data_modelling_demo.silver.scd2' exists. Performing SCD Type 2 MERGE...
✓ No new or changed records to insert

Silver table record count:
Total records in silver: 8
Current records (is_current=True): 6
Historical records: 2

Sample from silver table (showing current and historical):


customer_id,first_name,email,city,registration_date,last_updated_ts,_rescued_data,insert_timestamp,processed_timestamp,effective_start_date,effective_end_date,is_current
C001,John,john.smith@gmail.com,chicago,2024-01-10,2024-02-01 09:30:00,null,2026-06-25T12:22:41.805Z,2026-06-25T13:13:49.155Z,2026-06-25T13:13:49.155Z,null,true
C001,John,john.smith@gmail.com,new york,2024-01-10,2024-01-10 10:15:00,null,2026-06-24T12:09:30.082Z,2026-06-25T13:12:57.745Z,2026-06-25T13:12:57.745Z,2026-06-25T13:13:40.759Z,false
C002,Alice,alice123@gmail.com,boston,2024-01-11,2024-02-01 10:45:00,null,2026-06-25T12:22:41.805Z,2026-06-25T13:13:49.155Z,2026-06-25T13:13:49.155Z,null,true
C002,Alice,alice.johnson@gmail.com,boston,2024-01-11,2024-01-11 11:20:00,null,2026-06-24T12:09:30.082Z,2026-06-25T13:12:57.745Z,2026-06-25T13:12:57.745Z,2026-06-25T13:13:40.759Z,false
C003,Michael,michael.brown@gmail.com,chicago,2024-01-12,2024-01-12 09:10:00,null,2026-06-24T12:09:30.082Z,2026-06-25T13:12:57.745Z,2026-06-25T13:12:57.745Z,null,true
C004,Sarah,sarah.davis@gmail.com,seattle,2024-01-13,2024-01-13 15:45:00,null,2026-06-24T12:09:30.082Z,2026-06-25T13:12:57.745Z,2026-06-25T13:12:57.745Z,null,true
C005,David,david.wilson@gmail.com,dallas,2024-01-14,2024-01-14 16:00:00,null,2026-06-24T12:09:30.082Z,2026-06-25T13:12:57.745Z,2026-06-25T13:12:57.745Z,null,true
C006,Emma,emma.thomas@gmail.com,san francisco,2024-02-01,2024-02-01 11:15:00,null,2026-06-25T12:22:41.805Z,2026-06-25T13:13:49.155Z,2026-06-25T13:13:49.155Z,null,true


In [0]:
print(final_df.columns)

['customer_id', 'first_name', 'email', 'city', 'registration_date', 'last_updated_ts', '_rescued_data', 'insert_timestamp', 'processed_timestamp', 'effective_start_date', 'effective_end_date', 'is_current']
